# Dijet data distributions

Beam-orientation comparisons for configurable reconstructed-dijet selections. For the unflipped-Lab, flipped-Lab, and CM frames, plot common-scale Pb-going, p-going, and combined 2D maps, normalized pTave spectra with ratios to combined, and eta projections with ratios to combined. In the CM frame, also compare unnormalized forward/backward ratios and their ratios to the combined result.

All four triggers use the shared data-output resolver, which maps each configured selection to the combined, Pb-going, and p-going merged filenames.


Input resolution uses reusable files for the requested jet-selection stem, then the original ROOT file, and raises an error if neither exists. Source eta-cut tables must match the current producer; stale inputs require regeneration. `ETA_CUT_VALUE` (where present) is the physical per-jet acceptance, default 1.9; its index is derived from the shared table. `SINGLE_JET_BINNING` and `DIJET_BINNING` independently select existing `normal` or `test` intervals. Existing `PTAVE_BIN_SET` controls use `standard` as the name for normal binning. Trigger-specific normal bins remain the notebook defaults; test dijet bins use MB 50–100, Jet80 100–180, and Jet100 above 180 GeV. Jet60 has no test interval. Diagnostic frame scans and trigger-stitching overlap windows retain their established definitions.

<!-- detailed-workflow-guide -->

### Detailed workflow and data interpretation

Data histograms contain selected event or jet yields for explicit trigger, direction, and selection inputs. A projected bin content is the stored weighted count $N_i=\sum w_e$, with uncertainty inherited from the ROOT histogram. Trigger samples are not combined implicitly: each configured interval uses the documented trigger or stitching rule.

Unit-area normalization tests shape and removes luminosity and trigger-yield information. Density normalization additionally divides by bin width, $p_i=N_i/(\Delta x_i\sum_jN_j)$. Logarithmic display changes only rendering. Empty or prescaled regions should be diagnosed from the source trigger rather than interpreted as physical suppression.

## Environment and imports

This notebook locates the repository dynamically and imports PyROOT from the
active project environment. Start Jupyter from the repository root with
`py-env/bin/python -m jupyter notebook`; no machine-specific ROOT paths are
added at runtime.


In [ ]:
# Cell role: initialize the reproducible Python/ROOT environment and shared helpers.
# Interpretation: No physics histogram is modified here; ROOT ownership is configured before files open.
# The preceding Markdown gives the equations and physics assumptions for this step.
%load_ext autoreload
%autoreload 2

from pathlib import Path
from dataclasses import replace
import os

import sys

# Locate the repository without relying on a machine-specific absolute path.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "CMakeLists.txt").is_file()
        and (candidate / "hist_analysis").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Cannot locate the jetAnalysis repository. Start Jupyter from its root."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root

# Batch mode keeps plots reproducible and sends them to notebook/output files.
ROOT = load_root(batch=True)

from hist_analysis.python.notebook_setup import load_root

# Batch mode keeps plots reproducible and sends them to notebook/output files.
ROOT = load_root(batch=True)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.python.data_distributions import (
    draw_orientation_comparisons,
)
from hist_analysis.python.histogram_io import resolve_data_file
from hist_analysis.python.plot_labels import CMS_PRELIMINARY_HEADER, PPB_HEADER_RIGHT, PPB_MPLHEP_HEADER_RIGHT
from hist_analysis.python.root_style import (
    DEFAULT_PLOT_STYLE, configure_cms_header,
)
CMS_HEADER_LEFT = CMS_PRELIMINARY_HEADER
CMS_HEADER_RIGHT = PPB_HEADER_RIGHT
configure_cms_header(CMS_HEADER_LEFT, CMS_HEADER_RIGHT)

from hist_analysis.python.root_style import configure_mplhep_export

SAVE_MPLHEP = True  # retain ROOT output and optionally add mplhep PDFs
SAVE_MPLHEP_PNG = False
configure_mplhep_export(SAVE_MPLHEP, save_png=SAVE_MPLHEP_PNG)


In [ ]:
# Cell role: perform analysis step 2.
# Interpretation: Operations use the binning, normalization, and uncertainty conventions documented above.
# The preceding Markdown gives the equations and physics assumptions for this step.
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

In [ ]:
from hist_analysis.config.histograms import DIJET_PTAVE_BINS, trigger_dijet_bins
DIJET_BINNING = 'normal'  # normal or test; trigger-specific shared test assignment
# Cell role: define and validate user-facing analysis configuration.
# Interpretation: Changing these values can change inputs, selections, binning, normalization, or outputs.
# The preceding Markdown gives the equations and physics assumptions for this step.
DATA_DIR = Path(os.environ.get('PPB_DATA_DIR', BASE_DIR / 'exp'))
SELECTION = 'jetId'  # jetId, trkMax, or noSel
if SELECTION not in {'jetId', 'trkMax', 'noSel'}: raise ValueError(f'Unsupported SELECTION={SELECTION!r}')
OUTPUT_DIR = Path(os.environ.get('DATA_DIJET_OUTPUT_DIR', PROJECT_ROOT / 'hist_analysis/output/data_dijet_distributions')) / SELECTION
TRIGGERS = ('MinimumBias', 'Jet60', 'Jet80', 'Jet100')
DIRECTION_FILES = {trigger: {
    label: resolve_data_file(DATA_DIR, trigger, direction, SELECTION)
    for label, direction in (
        ('Pb-going', 'Pbgoing'), ('p-going', 'pgoing'), ('combined', 'combined'),
    )
} for trigger in TRIGGERS}
PTAVE_BINS = DIJET_PTAVE_BINS
# Established trigger handoffs; the shared high-pT interval also uses Jet100.
TRIGGER_PTAVE_RANGES = dict(zip(TRIGGERS, ((50, 80), (80, 100), (100, 120), (120, float('inf')))))
PTAVE_BINS_BY_TRIGGER = trigger_dijet_bins({
    trigger: [interval for interval in PTAVE_BINS if low <= interval[0] < interval[1] <= high]
    for trigger, (low, high) in TRIGGER_PTAVE_RANGES.items()
}, DIJET_BINNING)
if DIJET_BINNING == 'normal' and sorted(sum(PTAVE_BINS_BY_TRIGGER.values(), [])) != sorted(PTAVE_BINS):
    raise ValueError('Each standard pTave interval must belong to exactly one trigger range')
FRAMES = {
    'lab_unflipped': ('Lab unflipped', 'hRecoDijetPtEtaLabUnflipped', '#eta_{Lab,unflipped}^{dijet}'),
    'lab': ('Lab flipped', 'hRecoDijetPtEtaLab', '#eta_{Lab}^{dijet}'),
    'cm': ('CM', 'hRecoDijetPtEtaCM', '#eta_{CM}^{dijet}'),
}
REBIN_PT = 1; REBIN_ETA = 2; PT_DISPLAY_RANGE = (40.0, 500.0)
from hist_analysis.python.output_naming import nominal_jet_eta_cut
ETA_CUT = nominal_jet_eta_cut(PROJECT_ROOT)
ETA_DISPLAY_RANGE = (-ETA_CUT - 0.1, ETA_CUT + 0.1)
FB_DISPLAY_RANGE = (0.0, ETA_CUT + 0.1)
ETA_NORMALIZATION = 'integral'  # fractions per eta bin; 'bin_width' gives a density
PT_ORIENTATION_NORMALIZATION_RANGE = (110.0, 130.0)
ORIENTATION_RATIO_RANGE = (0.75, 1.25)
FB_RANGE = (0.5, 1.5); FB_ORIENTATION_RATIO_RANGE = (0.75, 1.25)
SAVE_PNG = False; DRAW_GRID = True
PLOT_STYLE = replace(DEFAULT_PLOT_STYLE, annotation_text_size=0.028, legend_text_size=0.028)
FULL_ETA_YLIM = (0.0, 0.15)  # None for automatic scaling
FB_YLIM = (0.75, 1.25)  # None for automatic scaling
GRID_FULL_ETA_YLIM = FULL_ETA_YLIM  # override independently; None for automatic
GRID_FB_YLIM = FB_YLIM  # override independently; None for a common automatic scale
SAVE_PLOTS = True  # final combined figures
OUTPUT_FORMATS = ['pdf', 'png']
DRAW_DATA_SYSTEMATIC = DIJET_BINNING == 'normal' and SELECTION == 'jetId'
DATA_SYSTEMATICS_DIR = Path(os.environ.get('DIJET_TOTAL_SYSTEMATICS_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis/output/systematics_total')) / SELECTION / DIJET_BINNING
DATA_SYSTEMATICS_CONFIGURATION = None  # optional exact configuration tag
missing = [str(path) for files in DIRECTION_FILES.values() for path in files.values() if not path.exists()]
if missing: raise FileNotFoundError('Missing ROOT files:\n' + '\n'.join(sorted(set(missing))))

OUTPUT_DIR = OUTPUT_DIR / DIJET_BINNING
COMBINED_PLOT_DIR = OUTPUT_DIR / 'mplhep' / 'combined'

# Output identity uses the actual configured selections. Each execution gets
# dated renderer directories and unique filenames; earlier files are preserved.
from hist_analysis.python.output_naming import OutputDirectory, OutputSelection
OUTPUT_SELECTIONS = OutputSelection(SELECTION, ETA_CUT, {'eta': PTAVE_BINS_BY_TRIGGER, 'spectrumNormalization': [PT_ORIENTATION_NORMALIZATION_RANGE]}, 'ptAve')
OUTPUT_DIR = OutputDirectory(OUTPUT_DIR, OUTPUT_SELECTIONS, workflow_root=Path(os.environ.get('DATA_DIJET_OUTPUT_DIR', PROJECT_ROOT / 'hist_analysis/output/data_dijet_distributions')))

# One deterministic ROOT archive; rerunning configuration starts a fresh file.
from hist_analysis.python.notebook_output import NotebookRootOutput
ROOT_OUTPUT = NotebookRootOutput(OUTPUT_DIR, '06_data_dijet_distributions')


## Pb-going and p-going overlays and ratios to combined

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
mb_orientation_results = draw_orientation_comparisons(
    'MinimumBias', DIRECTION_FILES['MinimumBias'], FRAMES, PTAVE_BINS_BY_TRIGGER['MinimumBias'],
    jet_kind='dijet', output_dir=OUTPUT_DIR, rebin_eta=REBIN_ETA, rebin_pt=REBIN_PT,
    pt_display_range=PT_DISPLAY_RANGE, pt_eta_range=None,
    pt_normalization_range=PT_ORIENTATION_NORMALIZATION_RANGE,
    selection=SELECTION, jet_eta_cut=ETA_CUT,
    ratio_range=ORIENTATION_RATIO_RANGE, save_png=SAVE_PNG,
    grid=DRAW_GRID, style=PLOT_STYLE, eta_normalization=ETA_NORMALIZATION,
    eta_display_range=ETA_DISPLAY_RANGE,
    fb_y_range=FB_RANGE, fb_x_range=FB_DISPLAY_RANGE,
    fb_ratio_range=FB_ORIENTATION_RATIO_RANGE,
)

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
jet60_orientation_results = draw_orientation_comparisons(
    'Jet60', DIRECTION_FILES['Jet60'], FRAMES, PTAVE_BINS_BY_TRIGGER['Jet60'],
    jet_kind='dijet', output_dir=OUTPUT_DIR, rebin_eta=REBIN_ETA, rebin_pt=REBIN_PT,
    pt_display_range=PT_DISPLAY_RANGE, pt_eta_range=None,
    pt_normalization_range=PT_ORIENTATION_NORMALIZATION_RANGE,
    selection=SELECTION, jet_eta_cut=ETA_CUT,
    ratio_range=ORIENTATION_RATIO_RANGE, save_png=SAVE_PNG,
    grid=DRAW_GRID, style=PLOT_STYLE, eta_normalization=ETA_NORMALIZATION,
    eta_display_range=ETA_DISPLAY_RANGE,
    fb_y_range=FB_RANGE, fb_x_range=FB_DISPLAY_RANGE,
    fb_ratio_range=FB_ORIENTATION_RATIO_RANGE,
)

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
jet80_orientation_results = draw_orientation_comparisons(
    'Jet80', DIRECTION_FILES['Jet80'], FRAMES, PTAVE_BINS_BY_TRIGGER['Jet80'],
    jet_kind='dijet', output_dir=OUTPUT_DIR, rebin_eta=REBIN_ETA, rebin_pt=REBIN_PT,
    pt_display_range=PT_DISPLAY_RANGE, pt_eta_range=None,
    pt_normalization_range=PT_ORIENTATION_NORMALIZATION_RANGE,
    selection=SELECTION, jet_eta_cut=ETA_CUT,
    ratio_range=ORIENTATION_RATIO_RANGE, save_png=SAVE_PNG,
    grid=DRAW_GRID, style=PLOT_STYLE, eta_normalization=ETA_NORMALIZATION,
    eta_display_range=ETA_DISPLAY_RANGE,
    fb_y_range=FB_RANGE, fb_x_range=FB_DISPLAY_RANGE,
    fb_ratio_range=FB_ORIENTATION_RATIO_RANGE,
)

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
jet100_orientation_results = draw_orientation_comparisons(
    'Jet100', DIRECTION_FILES['Jet100'], FRAMES, PTAVE_BINS_BY_TRIGGER['Jet100'],
    jet_kind='dijet', output_dir=OUTPUT_DIR, rebin_eta=REBIN_ETA, rebin_pt=REBIN_PT,
    pt_display_range=PT_DISPLAY_RANGE, pt_eta_range=None,
    pt_normalization_range=PT_ORIENTATION_NORMALIZATION_RANGE,
    selection=SELECTION, jet_eta_cut=ETA_CUT,
    ratio_range=ORIENTATION_RATIO_RANGE, save_png=SAVE_PNG,
    grid=DRAW_GRID, style=PLOT_STYLE, eta_normalization=ETA_NORMALIZATION,
    eta_display_range=ETA_DISPLAY_RANGE,
    fb_y_range=FB_RANGE, fb_x_range=FB_DISPLAY_RANGE,
    fb_ratio_range=FB_ORIENTATION_RATIO_RANGE,
)

## Final CM dijet figures

Reuse the combined trigger-specific projections. The default full-eta histogram is a fraction per bin. F/B uses the original unnormalized mirrored bins and independent ROOT errors. Statistical bars use the stored ROOT bin errors. For the default Jet ID standard bins, open brackets show the same provenance-checked relative totals used in notebook 07. Set `DRAW_DATA_SYSTEMATIC=False` for statistical bars alone. Save separate figures for each interval and paginate the 4×4 summaries in increasing pTave order.


In [ ]:
from hist_analysis.python.data_mc_plotting import load_data_total
from hist_analysis.python.dijet_data_plots import save_combined_dijet_figures

orientation_results = {
    'MinimumBias': mb_orientation_results,
    'Jet60': jet60_orientation_results,
    'Jet80': jet80_orientation_results,
    'Jet100': jet100_orientation_results,
}
combined_intervals = {}
combined_trigger = {}
for trigger, intervals in PTAVE_BINS_BY_TRIGGER.items():
    for interval in intervals:
        if interval in combined_intervals:
            raise ValueError(f'Duplicate pTave interval: {interval}')
        combined_trigger[interval] = 'mb' if trigger == 'MinimumBias' else trigger.lower()
        result = orientation_results[trigger][('cm', interval)]
        combined_intervals[interval] = {
            'eta': result['histograms']['combined'],
            'fb': result['forward_backward']['combined'],
        }

# Producer dijet selection (AnalysisContract.h); acceptance applies to both jets.
selection_labels = {'jetId': 'Jet ID', 'trkMax': 'Track maximum', 'noSel': 'No jet quality cut'}
jet_notes = (
    f"AK4 jets, {selection_labels[SELECTION]}",
    rf"$|\eta_{{\mathrm{{CM}}}}^{{\mathrm{{jet}}}}|\leq {ETA_CUT:g}$",
    r"$|\Delta\phi|\geq 2\pi/3$",
)
pt_notes = (r"$p_{\mathrm{T}}^{\mathrm{lead}}\geq 50$ GeV", r"$p_{\mathrm{T}}^{\mathrm{sublead}}\geq 40$ GeV")
combined_annotations = (", ".join(jet_notes[:2]), ", ".join((*pt_notes, jet_notes[2])))
combined_plot_files = []
for observable, individual_ylim, grid_ylim in (
    ('eta', FULL_ETA_YLIM, GRID_FULL_ETA_YLIM),
    ('fb', FB_YLIM, GRID_FB_YLIM),
):
    totals = {
        interval: load_data_total(
            DATA_SYSTEMATICS_DIR, interval,
            observable='full' if observable == 'eta' else 'fb',
            direction='combined', selection=SELECTION, binning=DIJET_BINNING,
            eta_cut=ETA_CUT, sample=combined_trigger[interval],
            configuration=DATA_SYSTEMATICS_CONFIGURATION,
        )
        for interval in combined_intervals
    } if SAVE_PLOTS and DRAW_DATA_SYSTEMATIC else None
    combined_plot_files.extend(save_combined_dijet_figures(
        {interval: entry[observable] for interval, entry in combined_intervals.items()},
        observable=observable, normalization=ETA_NORMALIZATION,
        output_dir=COMBINED_PLOT_DIR, selection=SELECTION, jet_eta_cut=ETA_CUT,
        annotations=combined_annotations, grid_notes=(jet_notes, pt_notes),
        cms_header_left=CMS_HEADER_LEFT, cms_header_right=PPB_MPLHEP_HEADER_RIGHT,
        individual_ylim=individual_ylim, grid_ylim=grid_ylim,
        totals=totals, save=SAVE_PLOTS, formats=OUTPUT_FORMATS,
    ))
print(f'Wrote {len(combined_plot_files)} combined plot files to {COMBINED_PLOT_DIR}')


### ROOT histogram archive
The file below includes plot-time histogram snapshots and retained results, with configuration metadata. Rerunning the configuration cell recreates the same file; rerunning plot cells replaces their keys.

In [ ]:
# Save retained numerical products in addition to exact plot-time snapshots.
NOTEBOOK_ROOT_FILE = ROOT_OUTPUT.finish(globals())
